# SkinSight — ResNet50 Skin Lesion Classifier (HAM10000)
### Reconstructed training pipeline — SGD (Nesterov) run, Phase 5 fine-tuning

This notebook reconstructs the training pipeline that produced `81__accuracy_sgd_phase_5.keras`
(~81% validation accuracy). Only the trained weights file survived, so this script was
rebuilt by inspecting the saved model's architecture and optimizer state directly, then
filling in the standard SkinSight pipeline (HAM10000, `preprocess_input`, progressive
layer unfreezing, class-weight capping).

**Confirmed directly from the saved model file:**
- Backbone: `ResNet50` (ImageNet weights), input `224x224x3`
- Head: `GlobalAveragePooling2D → Dropout(0.5) → Dense(256, relu) → Dropout(0.3) → Dense(7, softmax)`
- Phase 5 = full backbone unfrozen (all 108 weight layers trainable)
- Optimizer at save time: `SGD(learning_rate=0.00015, momentum=0.9, nesterov=True)`
- Loss: `categorical_crossentropy`

**Reconstructed (standard SkinSight pattern, adjust if your numbers differed):**
- Progressive unfreezing across 5 phases (head-only → increasing backbone depth → full unfreeze)
- `preprocess_input` (not `rescale=1./255`)
- Capped class weights to handle HAM10000's class imbalance (7 lesion classes, 10,015 images)

Works on **Kaggle or Colab** — the path-setup cell below detects which environment you're in.


In [ ]:
import os

# Detect environment (Kaggle vs Colab) and set the HAM10000 path accordingly
IN_KAGGLE = os.path.exists('/kaggle/input')
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_KAGGLE:
    # Adjust this to match the exact dataset slug you added on Kaggle
    DATA_DIR = '/kaggle/input/skin-cancer-mnist-ham10000'
    WORK_DIR = '/kaggle/working'
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this to wherever you kept the dataset in Drive
    DATA_DIR = '/content/drive/MyDrive/SkinSight/HAM10000'
    WORK_DIR = '/content'
else:
    DATA_DIR = './HAM10000'
    WORK_DIR = '.'

print('Environment:', 'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local')
print('DATA_DIR:', DATA_DIR)


In [ ]:
!pip install -q tensorflow scikit-learn pandas matplotlib


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 7  # akiec, bcc, bkl, df, mel, nv, vasc

tf.random.set_seed(42)
np.random.seed(42)


## 1. Load metadata and build train/val split

HAM10000 ships with `HAM10000_metadata.csv` mapping `image_id` → `dx` (diagnosis label)
across 7 classes, plus two image folders (`HAM10000_images_part_1` and `_part_2`).


In [ ]:
metadata = pd.read_csv(os.path.join(DATA_DIR, 'HAM10000_metadata.csv'))

# Map image_id -> full file path across the two image part-folders
part1_dir = os.path.join(DATA_DIR, 'HAM10000_images_part_1')
part2_dir = os.path.join(DATA_DIR, 'HAM10000_images_part_2')

def resolve_path(image_id):
    p1 = os.path.join(part1_dir, image_id + '.jpg')
    p2 = os.path.join(part2_dir, image_id + '.jpg')
    return p1 if os.path.exists(p1) else p2

metadata['path'] = metadata['image_id'].apply(resolve_path)

# 7 lesion classes
class_names = sorted(metadata['dx'].unique())
print('Classes:', class_names)
print(metadata['dx'].value_counts())

train_df, val_df = train_test_split(
    metadata, test_size=0.2, stratify=metadata['dx'], random_state=42
)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}')


## 2. Capped class weights

HAM10000 is heavily imbalanced (`nv` alone is ~67% of the dataset). Plain
`compute_class_weight` over-corrects and can destabilize training, so weights
are capped at a max ratio to keep the rarest class from dominating the loss.


In [ ]:
class_labels = sorted(train_df['dx'].unique())
raw_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(class_labels),
    y=train_df['dx']
)

MAX_WEIGHT_RATIO = 4.0  # cap: no class weighted more than 4x the minimum
capped_weights = np.clip(raw_weights, None, raw_weights.min() * MAX_WEIGHT_RATIO)

class_weight_dict = {i: w for i, w in enumerate(capped_weights)}
label_to_index = {label: i for i, label in enumerate(class_labels)}
print('Class weights (capped):')
for label, idx in label_to_index.items():
    print(f'  {label}: {class_weight_dict[idx]:.3f}')


## 3. Data generators

Uses ResNet50's `preprocess_input` (zero-centers per ImageNet channel stats) —
**not** `rescale=1./255`. This was a key fix in earlier iterations of this
project: `rescale` alone doesn't match what ResNet50 was pretrained on, and
hurt transfer-learning performance.


In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_dataframe(
    train_df, x_col='path', y_col='dx',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=class_labels, shuffle=True, seed=42
)

val_generator = val_datagen.flow_from_dataframe(
    val_df, x_col='path', y_col='dx',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=class_labels, shuffle=False
)


## 4. Build the model

Architecture confirmed from the saved `.keras` file's layer graph:
`ResNet50 (no top) → GAP → Dropout(0.5) → Dense(256, relu) → Dropout(0.3) → Dense(7, softmax)`


In [ ]:
def build_model():
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False  # start fully frozen — Phase 1

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)
    return model, base_model

model, base_model = build_model()
model.summary()


## 5. Progressive unfreezing — Phases 1 to 5

Each phase unfreezes more of the ResNet50 backbone and trains with a lower
learning rate, moving from Adam (head warm-up) to SGD+Nesterov for the final
fine-tuning phases — the SGD phases are what eventually outperformed the
Adam/AdamW runs (~81% vs ~75% val accuracy).

Layer counts below are ResNet50's conv-block boundaries under `stage 5 / stage 4 / stage 3`
(`conv5_block*`, `conv4_block*`, `conv3_block*`) — adjust the freeze index if your
version of the layer graph differs slightly.


In [ ]:
checkpoint_cb = ModelCheckpoint(
    os.path.join(WORK_DIR, 'best_model_{epoch:02d}_{val_accuracy:.2f}.keras'),
    monitor='val_accuracy', save_best_only=True, mode='max'
)
earlystop_cb = EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)
reduce_lr_cb = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)

callbacks = [checkpoint_cb, earlystop_cb, reduce_lr_cb]


In [ ]:
# ---- Phase 1: head only, backbone frozen ----
base_model.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])

history_p1 = model.fit(
    train_generator, validation_data=val_generator,
    epochs=10, class_weight=class_weight_dict, callbacks=callbacks
)


In [ ]:
# ---- Phase 2: unfreeze conv5_block (last ResNet stage) ----
base_model.trainable = True
for layer in base_model.layers:
    layer.trainable = layer.name.startswith('conv5_block')

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='categorical_crossentropy', metrics=['accuracy'])

history_p2 = model.fit(
    train_generator, validation_data=val_generator,
    epochs=10, class_weight=class_weight_dict, callbacks=callbacks
)


In [ ]:
# ---- Phase 3: unfreeze conv4_block + conv5_block ----
for layer in base_model.layers:
    layer.trainable = layer.name.startswith('conv4_block') or layer.name.startswith('conv5_block')

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])

history_p3 = model.fit(
    train_generator, validation_data=val_generator,
    epochs=10, class_weight=class_weight_dict, callbacks=callbacks
)


In [ ]:
# ---- Phase 4: unfreeze conv3_block + conv4_block + conv5_block, switch to SGD+Nesterov ----
for layer in base_model.layers:
    layer.trainable = (
        layer.name.startswith('conv3_block')
        or layer.name.startswith('conv4_block')
        or layer.name.startswith('conv5_block')
    )

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=5e-4, momentum=0.9, nesterov=True),
    loss='categorical_crossentropy', metrics=['accuracy']
)

history_p4 = model.fit(
    train_generator, validation_data=val_generator,
    epochs=10, class_weight=class_weight_dict, callbacks=callbacks
)


In [ ]:
# ---- Phase 5: full backbone unfrozen, low-LR SGD+Nesterov fine-tune ----
# Matches the saved checkpoint exactly: all 108 weight layers trainable,
# SGD(lr=0.00015, momentum=0.9, nesterov=True)
for layer in base_model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.00015, momentum=0.9, nesterov=True),
    loss='categorical_crossentropy', metrics=['accuracy']
)

history_p5 = model.fit(
    train_generator, validation_data=val_generator,
    epochs=15, class_weight=class_weight_dict, callbacks=callbacks
)

model.save(os.path.join(WORK_DIR, '81__accuracy_sgd_phase_5.keras'))
print('Saved final model.')


## 6. Evaluate



In [ ]:
val_loss, val_acc = model.evaluate(val_generator)
print(f'Validation accuracy: {val_acc:.4f}')
print(f'Validation loss: {val_loss:.4f}')


---
### Notes on this reconstruction

- The **architecture, input size, and Phase 5 optimizer settings are exact** —
  pulled directly from `81__accuracy_sgd_phase_5.keras`'s saved config.
- The **phase schedule (epochs, learning rates for phases 1–4, augmentation
  parameters, class-weight cap ratio)** are rebuilt to match the general
  SkinSight pipeline pattern (progressive unfreezing, `preprocess_input`,
  capped class weights) but weren't recoverable from the weights file itself —
  tune these if your actual run used different values.
- If you still have any Colab/Kaggle notebook version history, output logs,
  or a `history.pkl` / training log CSV lying around, those would let this
  be tightened up to match exactly.
